In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/
!git clone https://github.com/wangblog0/WeClone.git
%cd WeClone
!uv pip install --group main -e .
!uv pip install https://github.com/explosion/spacy-models/releases/download/zh_core_web_sm-3.8.0/zh_core_web_sm-3.8.0-py3-none-any.whl
!uv pip install flash-attn --no-build-isolation

In [ ]:
%cd /content/WeClone
!mkdir dataset/csv/
!mkdir dataset/csv/cyx/
!cp /content/drive/MyDrive/weclone/csv_data/cyx.csv dataset/csv/cyx/

In [ ]:
%cd /content/WeClone
!weclone-cli make-dataset

In [ ]:
import os
if "ORIGINAL_PATH" not in os.environ:
    os.environ["ORIGINAL_PATH"] = os.environ["PATH"]

In [ ]:
%cd /content/
!git clone https://github.com/wangblog0/LlamaFactory_colab.git
%cd LlamaFactory_colab
!git clone https://github.com/hiyouga/LlamaFactory.git
%cd LlamaFactory
!cp ../qwen3.5_35B_base.yaml ./
!uv venv --python 3.11
!uv pip install -e ".[metrics,bitsandbytes,qwen]" --python /content/LlamaFactory_colab/LlamaFactory/.venv/bin/python
!uv pip install flash-attn --no-build-isolation --python /content/LlamaFactory_colab/LlamaFactory/.venv/bin/python
!uv pip install "bitsandbytes>=0.39.0" --python /content/LlamaFactory_colab/LlamaFactory/.venv/bin/python
!uv pip install wandb --python /content/LlamaFactory_colab/LlamaFactory/.venv/bin/python
!/content/LlamaFactory_colab/LlamaFactory/.venv/bin/python -m wandb login

In [ ]:
!mkdir /content/LlamaFactory_colab/LlamaFactory/data/weclone_sft/
!cp /content/WeClone/dataset/res_csv/sft/dataset_info.json /content/LlamaFactory_colab/LlamaFactory/data/weclone_sft/
!cp /content/WeClone/dataset/res_csv/sft/sft-my.json /content/LlamaFactory_colab/LlamaFactory/data/weclone_sft/

In [ ]:
%cd /content/LlamaFactory_colab/LlamaFactory
!hf download Qwen/Qwen3.5-35B-A3B-Base --local-dir ./Qwen/Qwen3.5-35B-A3B-Base

In [ ]:
%cd /content/LlamaFactory_colab/LlamaFactory
%env MPLBACKEND=Agg
!/content/LlamaFactory_colab/LlamaFactory/.venv/bin/python -m llamafactory-cli train ./qwen3.5_35B_base.yaml

In [ ]:
%cd /content/LlamaFactory_colab/LlamaFactory
%env MPLBACKEND=Agg
!/content/LlamaFactory_colab/LlamaFactory/.venv/bin/python -m llamafactory-cli webui

In [ ]:
os.environ["PATH"] = os.environ["ORIGINAL_PATH"]

In [ ]:
from google.colab import output

# 将 8080 替换为你的服务端口
url = output.eval_js('google.colab.kernel.proxyPort(7860)')
print(f"你的服务传送门: {url}")

In [ ]:
from pathlib import Path
import shutil


SRC_DIR = Path("/content/LlamaFactory_colab/LlamaFactory/saves/qwen3.5-35b-a3b-base/weclone_lora_sft")
DST_DIR = Path("/content/drive/MyDrive/weclone/output_qwen3.5_35b_314")

# Minimal files needed for LoRA inference/deployment.
REQUIRED_FILES = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "processor_config.json",
    "chat_template.jinja",
]


def main() -> int:
    if not SRC_DIR.exists():
        print(f"[ERROR] Source directory does not exist: {SRC_DIR}")
        return 1

    DST_DIR.mkdir(parents=True, exist_ok=True)

    copied = []
    missing = []

    for name in REQUIRED_FILES:
        src_file = SRC_DIR / name
        dst_file = DST_DIR / name

        if src_file.exists():
            shutil.copy2(src_file, dst_file)
            copied.append(dst_file)
        else:
            missing.append(name)

    print("Copy finished")
    print(f"Source: {SRC_DIR}")
    print(f"Target: {DST_DIR}")
    print(f"Copied files: {len(copied)}")

    if copied:
        print("\nCopied:")
        for p in copied:
            size_mb = p.stat().st_size / (1024 * 1024)
            print(f"- {p} ({size_mb:.2f} MB)")

    if missing:
        print("\nMissing (skipped):")
        for n in missing:
            print(f"- {n}")

    return 0


if __name__ == "__main__":
    exit_code = main()
    if exit_code != 0:
        raise SystemExit(exit_code)


In [ ]:
!huggingface-cli login
!huggingface-cli repo create qwen3.5_35B_base_314

In [ ]:
from huggingface_hub import login, upload_folder

# (optional) Login with your Hugging Face credentials
# login()

# Push your model files
upload_folder(folder_path="/content/LlamaFactory_colab/LlamaFactory/model_export",
              repo_id="BriskLeopard/qwen3.5_35B_base_314",
              repo_type="model",
              revision="main"
              )


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# 给仓库的 main 分支打一个 v1.0 的标签
api.create_tag(
    repo_id="你的用户名/你的仓库名",
    tag="v1.0",
    tag_message="这是微调的第一版 72B 模型"
)
print("Tag 创建成功！")